# RNN: quando a ordem carrega a informação

**Para quem está começando.** Este notebook supõe que você já viu o MLP, da mesma pasta. A ideia é rodar de cima para baixo, célula por célula, e olhar os números.

O que vamos fazer, em ordem:

1. criar um problema em que a resposta depende da ordem dos elementos;
2. comparar um modelo que ignora a ordem com uma rede recorrente;
3. testar a mesma rede em duas tarefas, uma de memória e outra de combinação;
4. entender por que existem LSTM e GRU.

Roda no Google Colab, que já traz o PyTorch. No Jupyter do seu computador, vale o ambiente do `README.md` (a versão usada aqui é `torch==2.13.0`).


## O que a recorrência faz de diferente

Uma camada densa ou convolucional recebe uma entrada de tamanho fixo. A recorrente percorre a entrada **um elemento por vez** e carrega consigo um estado escondido, que é reescrito a cada passo. É esse estado que funciona como memória: no passo 5, ele já viu os quatro elementos anteriores.

Duas consequências práticas:

- **os pesos são os mesmos em todos os passos**, então uma RNN aceita sequência de qualquer comprimento e não ganha parâmetros quando a sequência cresce;
- **a informação caminha passo a passo**, então a ordem dos elementos muda o resultado, e a média ou a soma deles não bastam para decidir.

O outro lado da moeda está no treino. Como o erro volta pela sequência inteira, o gradiente que chega ao começo é um produto de muitos fatores, e ele pode sumir (ou explodir) conforme a sequência cresce. É o problema do gradiente que desvanece, e é uma das razões pelas quais LSTM e GRU existem. Este notebook mede quando isso realmente atrapalha, e a resposta surpreende.


## Passo 0: ambiente e configuração

A semente é fixada em `numpy` e em `torch`, e as funções de treino são as mesmas dos outros notebooks da pasta, adaptadas para entrada em sequência.


In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (9, 4)
print(f"✅ torch {torch.__version__} em {device} | seed {RANDOM_STATE}")

## Passo 1: um problema em que a ordem decide

Cada sequência tem 10 elementos, cada um valendo 0 ou 1, sorteados ao acaso. A classe é 1 quando **o primeiro e o último elementos são diferentes**, e 0 quando são iguais.

Repare no que isso faz com um modelo que ignora a ordem: a soma da sequência não distingue as duas classes, e a regressão logística treinada só na soma fica perto de 50%. A informação existe, mas está na ordem, e não na contagem.


In [ ]:
N_TREINO, N_TESTE = 1000, 400


def make_sequences(n: int, length: int, tarefa: str = "extremos"):
    """Random 0/1 sequences; the label depends on the order of the elements."""
    X = rng.integers(0, 2, size=(n, length)).astype(np.float32)
    if tarefa == "extremos":
        y = (X[:, 0] != X[:, -1]).astype(np.float32)
    else:                                  # paridade: o numero de 1s e impar
        y = (X.sum(axis=1) % 2 == 1).astype(np.float32)
    return X, y


X_treino, y_treino = make_sequences(N_TREINO, 10)
X_teste, y_teste = make_sequences(N_TESTE, 10)
print(f"🔎 treino {X_treino.shape} | teste {X_teste.shape} | "
      f"positivos no treino {y_treino.mean():.3f}")

sem_ordem = LogisticRegression().fit(X_treino.sum(axis=1, keepdims=True), y_treino)
acuracia_sem_ordem = float((sem_ordem.predict(X_teste.sum(axis=1, keepdims=True)) == y_teste).mean())
print(f"📊 só com a soma dos elementos: acurácia = {acuracia_sem_ordem:.4f}")

exemplos = pd.DataFrame(X_treino[:4].astype(int))
exemplos.columns = [f"t{i + 1}" for i in range(10)]
exemplos["classe"] = y_treino[:4].astype(int)
exemplos["soma"] = X_treino[:4].sum(axis=1).astype(int)
display(exemplos.set_index("soma"))

## Passo 2: a rede que carrega estado

A RNN abaixo tem uma camada recorrente de 32 unidades e um cabeçalho denso que transforma o estado final num único número. Note que o `input_size` é 1: cada passo recebe um elemento da sequência, e o comprimento entra na forma do tensor, não na arquitetura.

A avaliação também muda de formato: em vez de uma tabela, a entrada é `(exemplos, comprimento, 1)`.


In [ ]:
class RNN(nn.Module):
    """Recurrent layer over the sequence, then a dense head on the last state."""

    def __init__(self, hidden: int = 32):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=hidden, batch_first=True)
        self.cabeca = nn.Linear(hidden, 1)

    def forward(self, x):
        saida, estado = self.rnn(x)
        return self.cabeca(estado[-1])


def to_tensor(X, y=None):
    X_t = torch.tensor(np.asarray(X, dtype=np.float32)).unsqueeze(-1)
    if y is None:
        return X_t
    return X_t, torch.tensor(np.asarray(y, dtype=np.float32)).view(-1, 1)


def predict_proba(modelo: nn.Module, X) -> np.ndarray:
    modelo.eval()
    with torch.no_grad():
        return torch.sigmoid(modelo(to_tensor(X))).numpy().ravel()


def train_model(modelo: nn.Module, X, y, epochs: int = 300, lr: float = 0.01):
    """Full-batch training on the whole sequence at once; returns the loss per epoch."""
    X_t, y_t = to_tensor(X, y)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    perdas = []
    for _ in range(epochs):
        modelo.train()
        otimizador.zero_grad()
        loss = loss_fn(modelo(X_t), y_t)
        loss.backward()
        otimizador.step()
        perdas.append(float(loss.item()))
    return perdas


def acuracia(modelo: nn.Module, X, y) -> float:
    return float(((predict_proba(modelo, X) > 0.5) == y).mean())

In [ ]:
torch.manual_seed(RANDOM_STATE)
rede = RNN(hidden=32)
inicio = time.perf_counter()
perdas = train_model(rede, X_treino, y_treino, epochs=300)
print(f"⏱️ 300 épocas em {time.perf_counter() - inicio:.1f}s")
print(f"parâmetros da RNN: {sum(p.numel() for p in rede.parameters()):,}")

fig, ax = plt.subplots()
ax.plot(perdas, color="#0F6B68")
ax.set_xlabel("época")
ax.set_ylabel("perda (BCE)")
ax.set_title("Curva de perda do treino")
plt.show()

pd.DataFrame([
    {"modelo": "Soma dos elementos", "acuracia_teste": acuracia_sem_ordem},
    {"modelo": "RNN", "acuracia_teste": acuracia(rede, X_teste, y_teste)},
]).set_index("modelo").round(4).pipe(display)

### Como ler isso

O modelo que só vê a soma fica no chute: as duas classes têm a mesma soma média, então não há o que aprender ali. A RNN resolve porque o estado escondido chega ao último passo ainda carregando o primeiro elemento, e é a comparação entre os dois que define a classe.

Isso também é o argumento a favor dela em sequência de comprimento variável: os mesmos 1 153 parâmetros servem para sequência de 5 ou de 500 elementos, sem crescer. Uma camada densa sobre a sequência achatada precisaria de um peso para cada posição, e não aceitaria outro comprimento sem trocar de arquitetura.

O contra está no tempo: 300 épocas custaram mais que as 300 do MLP, porque o cálculo é sequencial, passo a passo. Em GPU a diferença diminui, mas o formato sequencial continua sendo o gargalo.


## Passo 3: a mesma rede, sequências cada vez mais longas

Se o gradiente que desvanece é o problema, a conta é simples: quanto mais longa a sequência, mais difícil. Vamos medir. A rede treinada acima é avaliada em sequências de até 640 elementos, sem nenhum retreino: mesmos pesos, mesma tarefa.


In [ ]:
comprimentos = [10, 20, 40, 80, 160, 320, 640]
linhas = []
for comprimento in comprimentos:
    X_aval, y_aval = make_sequences(N_TESTE, comprimento)
    linhas.append({"comprimento": comprimento, "acuracia": acuracia(rede, X_aval, y_aval)})

tabela_comprimento = pd.DataFrame(linhas).set_index("comprimento")
display(tabela_comprimento.round(4))

fig, ax = plt.subplots()
ax.plot(tabela_comprimento.index, tabela_comprimento["acuracia"], marker="o", color="#0F6B68")
ax.axhline(0.5, color="#B45309", linestyle="--", linewidth=1, label="chute")
ax.set_xlabel("comprimento da sequência")
ax.set_ylabel("acurácia")
ax.set_title("Tarefa de memória: a rede aguenta sequências longas")
ax.set_xscale("log")
ax.legend()
plt.show()

## Passo 4: a tarefa que ela não aprende

Mesma arquitetura, mesma receita de treino, mesma semente. A única coisa que muda é a tarefa: agora a classe é 1 quando **o número de 1s na sequência é ímpar**.

Guardar um valor até o fim, que era o que a tarefa anterior pedia, é fácil para um estado escondido: basta manter uma direção no espaço de estados. Paridade é outra coisa. Ela exige combinar a informação de todos os passos numa função que não é soma, e o estado tem de carregar o resultado parcial a cada passo.


In [ ]:
X_par, y_par = make_sequences(N_TREINO, 10, tarefa="paridade")
X_par_teste, y_par_teste = make_sequences(N_TESTE, 10, tarefa="paridade")

torch.manual_seed(RANDOM_STATE)
rede_paridade = RNN(hidden=32)
train_model(rede_paridade, X_par, y_par, epochs=300)

print(f"acuracia no treino (L=10): {acuracia(rede_paridade, X_par, y_par):.4f}")
linhas = []
for comprimento in comprimentos:
    X_aval, y_aval = make_sequences(N_TESTE, comprimento, tarefa="paridade")
    linhas.append({"comprimento": comprimento, "acuracia": acuracia(rede_paridade, X_aval, y_aval)})

tabela_paridade = pd.DataFrame(linhas).set_index("comprimento")
display(tabela_paridade.round(4))

### Como ler isso

Os dois resultados juntos dizem mais que a regra que se repete por aí:

- na **tarefa de memória**, a RNN simples aguentou sequências longas sem perder acurácia. O gradiente atravessou centenas de passos, e o estado deu conta de segurar um bit até o fim;
- na **tarefa de combinação**, ela não aprendeu nem no comprimento em que foi treinada: acerta a maior parte do treino e, no teste do mesmo comprimento, cai bem abaixo disso, voltando ao chute nas sequências maiores. É o sintoma clássico de decorar as sequências em vez de aprender a regra.

Ou seja: o comprimento, sozinho, não é o vilão. O que decide é a **operação** que a tarefa exige do estado. Guardar um valor é uma operação estável, e o gradiente atravessa; combinar todos os passos numa função não linear, passo a passo, é uma operação que a célula recorrente simples não consegue representar nem treinar bem. O trabalho de Bengio e colaboradores (1994) formula exatamente isso, e é o motivo pelo qual a resposta da área foi mudar a célula, não alongar o treino.

O remédio conhecido, na ordem em que se tenta: cortar o gradiente num limite (`clip_grad_norm_`), que resolve a explosão e não a atenuação; e trocar a célula por LSTM ou GRU, que têm portas para decidir o que guardar e o que descartar. Aqui é uma linha: `nn.RNN` vira `nn.LSTM`, e o estado passa a ser uma tupla. É o experimento que eu faria em seguida, com a tarefa de paridade, comparando as três células.

## Resumo

A RNN processa a entrada passo a passo e carrega um estado, então a ordem dos elementos muda o resultado e o comprimento da sequência não precisa ser fixo, sem custo extra de parâmetros. O preço é o treino: a informação atravessa a sequência inteira em cada passo, e o erro precisa voltar pelo mesmo caminho.

O que este notebook mediu, e que a regra repetida não conta: sequência longa não é automaticamente problema. Uma tarefa de memória atravessou 640 passos intacta; uma tarefa de combinação falhou já no comprimento do treino, e falhou por representação, não por comprimento. Antes de culpar o gradiente que desvanece, olhe a tarefa e veja o que ela exige do estado.

**Próximo passo:** a mesma família de problema, resolvida com atenção que olha todos os passos ao mesmo tempo em vez de carregar estado, é o que o transformer faz.

### Referências

**A célula recorrente e a memória**

Elman, J. L. (1990). *Finding structure in time.* Cognitive Science 14(2), 179-211.

Hochreiter, S., & Schmidhuber, J. (1997). *Long short-term memory.* Neural Computation 9(8), 1735-1780.

**O problema do gradiente**

Bengio, Y., Simard, P., & Frasconi, P. (1994). *Learning long-term dependencies with gradient descent is difficult.* IEEE Transactions on Neural Networks 5(2), 157-166.

```bibtex
@article{elman1990finding,
  title   = {Finding structure in time},
  author  = {Elman, Jeffrey L.},
  journal = {Cognitive Science},
  volume  = {14},
  number  = {2},
  pages   = {179--211},
  year    = {1990}
}

@article{hochreiter1997long,
  title   = {Long short-term memory},
  author  = {Hochreiter, Sepp and Schmidhuber, J{\"u}rgen},
  journal = {Neural Computation},
  volume  = {9},
  number  = {8},
  pages   = {1735--1780},
  year    = {1997}
}

@article{bengio1994learning,
  title   = {Learning long-term dependencies with gradient descent is difficult},
  author  = {Bengio, Yoshua and Simard, Patrice and Frasconi, Paolo},
  journal = {IEEE Transactions on Neural Networks},
  volume  = {5},
  number  = {2},
  pages   = {157--166},
  year    = {1994}
}
```
